# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [68]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [69]:
# Load the dataset (adjust the path to the correct location
path = "AviationData.csv"
df = pd.read_csv(path, encoding="latin-1")

df.head()

C:\Users\Omistaja\AppData\Local\Temp\ipykernel_8604\3596225010.py:3: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, encoding="latin-1")


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [70]:
df.shape
#to check rows and columns

(88889, 31)

In [71]:
df.info 
#To check the  total number of rows and columns.
# each column name and its data type (object, int64, float64, etc.).
#How many non‑null (non‑missing) values each column has.

<bound method DataFrame.info of              Event.Id Investigation.Type Accident.Number  Event.Date  \
0      20001218X45444           Accident      SEA87LA080  1948-10-24   
1      20001218X45447           Accident      LAX94LA336  1962-07-19   
2      20061025X01555           Accident      NYC07LA005  1974-08-30   
3      20001218X45448           Accident      LAX96LA321  1977-06-19   
4      20041105X01764           Accident      CHI79FA064  1979-08-02   
...               ...                ...             ...         ...   
88884  20221227106491           Accident      ERA23LA093  2022-12-26   
88885  20221227106494           Accident      ERA23LA095  2022-12-26   
88886  20221227106497           Accident      WPR23LA075  2022-12-26   
88887  20221227106498           Accident      WPR23LA076  2022-12-26   
88888  20221230106513           Accident      ERA23LA097  2022-12-29   

              Location        Country   Latitude  Longitude Airport.Code  \
0      MOOSE CREEK, ID  Uni

In [72]:
df.describe
# to understand the distribution of values.

<bound method NDFrame.describe of              Event.Id Investigation.Type Accident.Number  Event.Date  \
0      20001218X45444           Accident      SEA87LA080  1948-10-24   
1      20001218X45447           Accident      LAX94LA336  1962-07-19   
2      20061025X01555           Accident      NYC07LA005  1974-08-30   
3      20001218X45448           Accident      LAX96LA321  1977-06-19   
4      20041105X01764           Accident      CHI79FA064  1979-08-02   
...               ...                ...             ...         ...   
88884  20221227106491           Accident      ERA23LA093  2022-12-26   
88885  20221227106494           Accident      ERA23LA095  2022-12-26   
88886  20221227106497           Accident      WPR23LA075  2022-12-26   
88887  20221227106498           Accident      WPR23LA076  2022-12-26   
88888  20221230106513           Accident      ERA23LA097  2022-12-29   

              Location        Country   Latitude  Longitude Airport.Code  \
0      MOOSE CREEK, ID  U

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [73]:
for col in df.columns:
    if any(keyword in col for keyword in
           ['Make','Model','Aircraft','Purpose','Fatal','Serious','Damage']):
        print(col)

Aircraft.damage
Aircraft.Category
Make
Model
Purpose.of.flight
Total.Fatal.Injuries
Total.Serious.Injuries


In [74]:
df[['Aircraft.Category', 'Amateur.Built', 'Event.Date']].info()
df[['Aircraft.Category', 'Amateur.Built']].value_counts().head(10)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Aircraft.Category  32287 non-null  object
 1   Amateur.Built      88787 non-null  object
 2   Event.Date         88889 non-null  object
dtypes: object(3)
memory usage: 2.0+ MB


Aircraft.Category  Amateur.Built
Airplane           No               24417
Helicopter         No                3295
Airplane           Yes               3183
Glider             No                 476
Balloon            No                 229
Helicopter         Yes                143
Gyrocraft          Yes                141
Weight-Shift       No                 139
Powered Parachute  No                  83
Gyrocraft          No                  32
Name: count, dtype: int64

In [75]:
#Standardize text and handle missing values.
# Standardize Aircraft.Category
df['Aircraft.Category'] = (
    df['Aircraft.Category']
    .astype(str)
    .str.strip()
    .str.title()
    .replace(['Nan', 'Na', 'None', ''], 'Unknown')
)

# Standardize Amateur.Built
df['Amateur.Built'] = (
    df['Amateur.Built']
    .astype(str)
    .str.strip()
    .str.title()
)


If "Unknown" remains in Aircraft.Category, we’ll likely exclude it since client only wants known airplane types.
For Amateur.Built, anything other than "Yes" is treated as professional ("No" or missing).

In [76]:
#Filtering to include only the aircraft relevant to the client
#Keep only airplanes
df = df[
    df['Aircraft.Category'].str.contains('Airplane', case=False, na=False)
]


. Exclude amateur‑built aircraft

Keep only records from the last 40 years
 retain events from 1986 onwards.

In [77]:


df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
df = df[df['Event.Date'] >= '1986-01-01']


C:\Users\Omistaja\AppData\Local\Temp\ipykernel_8604\1492746633.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')


In [78]:
df = df[df['Amateur.Built'] != 'Yes']


checking the filtered data

In [79]:
print("Filtered shape:", df.shape)
df[['Aircraft.Category', 'Amateur.Built', 'Event.Date']].sample(5)


Filtered shape: (21440, 31)


,Aircraft.Category,Amateur.Built,Event.Date
51987,Airplane,No,2002-01-10
80058,Airplane,No,2017-06-14
75854,Airplane,No,2014-09-28
74889,Airplane,No,2014-03-31
87182,Airplane,No,2021-11-17


In [80]:
#inspecting relevant columns
df.columns.to_list()
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 21440 entries, 14259 to 88886
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                21440 non-null  object        
 1   Investigation.Type      21440 non-null  object        
 2   Accident.Number         21440 non-null  object        
 3   Event.Date              21440 non-null  datetime64[ns]
 4   Location                21433 non-null  object        
 5   Country                 21439 non-null  object        
 6   Latitude                19173 non-null  object        
 7   Longitude               19167 non-null  object        
 8   Airport.Code            13970 non-null  object        
 9   Airport.Name            14056 non-null  object        
 10  Injury.Severity         20627 non-null  object        
 11  Aircraft.damage         20211 non-null  object        
 12  Aircraft.Category       21440 non-null  object 

In [81]:
#data Cleaning
#Handling missing vales.
#counts how many True (i.e. missing) values there are in each column.
#.sort_values(ascending=False) → shows the columns ordered from most missing values to least.
df.isna().sum().sort_values(ascending=False)


Schedule                  18916
Broad.phase.of.flight     18633
Air.carrier               11281
Airport.Code               7470
Airport.Name               7384
Report.Status              4663
Engine.Type                3967
Purpose.of.flight          3691
Weather.Condition          2990
Total.Serious.Injuries     2825
Total.Fatal.Injuries       2748
Number.of.Engines          2562
Total.Minor.Injuries       2538
Longitude                  2273
Latitude                   2267
Aircraft.damage            1229
Publication.Date            966
Injury.Severity             813
Total.Uninjured             714
FAR.Description             480
Registration.Number         216
Model                        18
Location                      7
Make                          3
Country                       1
Aircraft.Category             0
Event.Date                    0
Accident.Number               0
Investigation.Type            0
Event.Id                      0
Amateur.Built                 0
dtype: i

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [82]:
# Selecting and renaming key columns for ease
df.rename(columns={
    'Total.Fatal.Injuries': 'Fatal_Injuries',
    'Total.Serious.Injuries': 'Serious_Injuries',
    'Aircraft.damage': 'Aircraft_Damage'
}, inplace=True)

In [83]:
# Replace missing injury counts with 0, assuming blank = none reported
df['Fatal_Injuries'] = df['Fatal_Injuries'].fillna(0)
df['Serious_Injuries'] = df['Serious_Injuries'].fillna(0)

In [84]:
## Standardize Aircraft_Damage and filliing missing values
df['Aircraft_Damage'] = df['Aircraft_Damage'].str.strip().str.title()
df['Aircraft_Damage'] = df['Aircraft_Damage'].fillna('Unknown')

#total‑injury and survival 

In [85]:
# Total serious + fatal
df['Total_Severe_Injuries'] = (
    df['Fatal_Injuries'] + df['Serious_Injuries']
)


Quantify the aircraft’s resistance to total destruction.

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [86]:
df.rename(columns=lambda x: x.strip())

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Air.carrier,Fatal_Injuries,Serious_Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total_Severe_Injuries
14259,20001213X33054,Accident,FTW86FA050,1986-03-29,"SEAGOVILLE, TX",United States,NaN,NaN,59F,SEAGOVILLE,...,NaN,3.0,2.0,NaN,NaN,VMC,Takeoff,Probable Cause,17-10-2016,5.0
14357,20001213X33306,Accident,LAX86LA166,1986-04-08,"MESA, AZ",United States,NaN,NaN,FFZ,FALCON FIELD,...,NaN,0.0,0.0,NaN,2.0,VMC,Landing,Probable Cause,03-08-2011,0.0
14420,20001213X33276,Accident,FTW86FA066B,1986-04-15,"HANKAMER, TX",United States,NaN,NaN,NaN,NaN,...,NaN,2.0,0.0,NaN,NaN,VMC,Descent,Probable Cause,08-04-2013,2.0
14421,20001213X33276,Accident,FTW86FA066A,1986-04-15,"HANKAMER, TX",United States,NaN,NaN,NaN,NaN,...,NaN,2.0,0.0,NaN,NaN,VMC,Standing,Probable Cause,08-04-2013,2.0
14711,20001213X33697,Accident,SEA86FA121B,1986-05-19,"MEAD, WA",United States,NaN,NaN,NaN,NaN,...,NaN,2.0,0.0,1.0,NaN,VMC,Descent,Probable Cause,17-10-2016,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88869,20221213106455,Accident,WPR23LA065,2022-12-13,"Lewistown, MT",United States,047257N,0109280W,KLWT,Lewiston Municipal Airport,...,NaN,0.0,0.0,0.0,1.0,NaN,NaN,NaN,14-12-2022,0.0
88873,20221215106463,Accident,ERA23LA090,2022-12-14,"San Juan, PR",United States,182724N,0066554W,SIG,FERNANDO LUIS RIBAS DOMINICCI,...,SKY WEST AVIATION INC TRUSTEE,0.0,0.0,0.0,1.0,VMC,NaN,NaN,27-12-2022,0.0
88876,20221219106475,Accident,WPR23LA069,2022-12-15,"Wichita, KS",United States,373829N,0972635W,ICT,WICHITA DWIGHT D EISENHOWER NT,...,NaN,0.0,0.0,0.0,1.0,NaN,NaN,NaN,19-12-2022,0.0
88877,20221219106470,Accident,ERA23LA091,2022-12-16,"Brooksville, FL",United States,282825N,0822719W,BKV,BROOKSVILLE-TAMPA BAY RGNL,...,GERBER RICHARD E,0.0,1.0,0.0,0.0,VMC,NaN,NaN,23-12-2022,1.0


In [87]:
# Show all column names exactly as read
print(df.columns.tolist())


['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft_Damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Fatal_Injuries', 'Serious_Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date', 'Total_Severe_Injuries']


In [88]:
df.loc[:, 'Aircraft_Damage'] = (
    df['Aircraft_Damage']
    .astype(str)
    .str.strip()
    .str.title()
    .replace(['Nan', 'Na', 'None', ''], 'Unknown')
)


destroyed flag created to mark whether an aircraft was completely destroyed in an accident or not

In [89]:
df['Destroyed_Flag'] = df['Aircraft_Damage'].apply(
    lambda x: 1 if x == 'Destroyed' else 0
)


In [90]:
df[['Aircraft_Damage', 'Destroyed_Flag']].value_counts().head()


Aircraft_Damage  Destroyed_Flag
Substantial      0                 16985
Destroyed        1                  2311
Unknown          0                  1326
Minor            0                   818
Name: count, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

investigating the make column

In [91]:
print(df['Make'].value_counts().head(20))
print("\nTotal unique Makes:", df['Make'].nunique())
print("\nMissing values:", df['Make'].isna().sum())

Make
CESSNA                4867
PIPER                 2803
Cessna                2273
Piper                 1186
BOEING                1037
BEECH                 1018
Beech                  412
MOONEY                 238
Boeing                 231
CIRRUS DESIGN CORP     218
AIR TRACTOR INC        217
AIRBUS                 215
BELLANCA               158
AERONCA                149
MAULE                  144
Mooney                 125
EMBRAER                123
Air Tractor            117
LUSCOMBE                95
DEHAVILLAND             91
Name: count, dtype: int64

Total unique Makes: 1332

Missing values: 3


In [92]:
## Check for inconsistencies
print(df['Make'].str.upper().value_counts().head(20))

Make
CESSNA                7140
PIPER                 3989
BEECH                 1430
BOEING                1268
MOONEY                 363
AIRBUS                 244
CIRRUS DESIGN CORP     220
BELLANCA               219
AIR TRACTOR INC        219
MAULE                  215
AIR TRACTOR            206
AERONCA                200
CHAMPION               158
EMBRAER                153
GRUMMAN                147
LUSCOMBE               141
CIRRUS                 137
STINSON                129
MCDONNELL DOUGLAS      108
NORTH AMERICAN         106
Name: count, dtype: int64


## Make Column - Cleaning Tasks

1. **Inconsistent casing** - e.g. 'cessna', 'Cessna', 'CESSNA' all refer to the same make
2. **Leading/trailing whitespace** - extra spaces causing duplicates
3. **Missing values** - nulls need to be dropped
4. **Low frequency makes**- keeps only makes with 50+ occurrences to ensure meaningful analysis

EXECUTING CLEANING

In [93]:
# capitalization and strip whitespace
df['Make'] = (
    df['Make']
    .astype(str)
    .str.strip()
    .str.title()
)
 #Replace missing values
#  Unknown makes are labelled 'Unknown' rather than dropped
df['Make'] = df['Make'].replace('Nan', 'Unknown')

# Check cleaned values
print("After cleaning:")
print(df['Make'].value_counts().head(20))

After cleaning:
Make
Cessna                7140
Piper                 3989
Beech                 1430
Boeing                1268
Mooney                 363
Airbus                 244
Cirrus Design Corp     220
Bellanca               219
Air Tractor Inc        219
Maule                  215
Air Tractor            206
Aeronca                200
Champion               158
Embraer                153
Grumman                147
Luscombe               141
Cirrus                 137
Stinson                129
Mcdonnell Douglas      108
North American         106
Name: count, dtype: int64


In [94]:
# Keep only Makes with 50+ occurrences
# Makes with fewer than 50 records don't provide enough data
# for reliable analysis and could skew results
make_counts = df['Make'].value_counts()
valid_makes  = make_counts[make_counts >= 50].index

df = df[df['Make'].isin(valid_makes)].copy()

print(f"Makes remaining after threshold filter: {df['Make'].nunique()}")
print(f"Rows remaining: {df.shape[0]}")
print("\nRemaining Makes:")
print(df['Make'].value_counts())

Makes remaining after threshold filter: 36
Rows remaining: 17889

Remaining Makes:
Make
Cessna                            7140
Piper                             3989
Beech                             1430
Boeing                            1268
Mooney                             363
Airbus                             244
Cirrus Design Corp                 220
Bellanca                           219
Air Tractor Inc                    219
Maule                              215
Air Tractor                        206
Aeronca                            200
Champion                           158
Embraer                            153
Grumman                            147
Luscombe                           141
Cirrus                             137
Stinson                            129
Mcdonnell Douglas                  108
North American                     106
Dehavilland                         95
Taylorcraft                         93
Aero Commander                      90
Aviat Aircraft 

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [95]:
#GET RID OF ANY NaNs
# Drop rows with missing model information
df = df.dropna(subset=['Model'])

# Remove accidental blank strings
df = df[df['Model'].astype(str).str.strip() != '']


In [96]:
#Inspect the columns counts for each model
# Check how many models per manufacturer
model_counts = df['Model'].value_counts().head(10)
print(model_counts)

# Check uniqueness of model names across makes
dup_models = (df.groupby('Model')['Make']
                .nunique()
                .sort_values(ascending=False)
                .head(10))
print("Models linked to multiple makes:\n", dup_models)


Model
172     769
737     403
152     315
182     304
172S    276
PA28    273
172N    249
SR22    240
180     213
A36     181
Name: count, dtype: int64
Models linked to multiple makes:
 Model
S2R      3
7GCBC    3
500      3
400      3
7GCAA    3
7ECA     3
8KCAB    3
7AC      3
7EC      3
8GCBC    3
Name: Make, dtype: int64


In [97]:
#Create a unique identifier for each plane type
#If Model names aren’t unique by make, combine the two.
df['Make_Model'] = (
    df['Make'].astype(str).str.strip() + " " +
    df['Model'].astype(str).str.strip()
)


In [98]:
#verification.
print(df['Make_Model'].nunique(), "unique make‑model combinations")
df[['Make', 'Model', 'Make_Model']].sample(5)


2164 unique make‑model combinations


,Make,Model,Make_Model
60888,Cessna,172S,Cessna 172S
68093,Cessna,T206H,Cessna T206H
86311,Beech,A36,Beech A36
66214,Maule,M-5-235C,Maule M-5-235C
79465,Beech,C 99,Beech C 99


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [99]:
#check missing values here
cols_to_check = [
    'Engine.Type',
    'Weather.Condition',
    'Number.of.Engines',
    'Purpose.of.flight',
    'Broad.phase.of.flight'
]

for col in cols_to_check:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(10))



--- Engine.Type ---
Engine.Type
Reciprocating      12827
NaN                 3222
Turbo Prop           927
Turbo Fan            701
Unknown              106
Turbo Jet             71
Geared Turbofan       12
Turbo Shaft            9
UNK                    1
Name: count, dtype: int64

--- Weather.Condition ---
Weather.Condition
VMC    14283
NaN     2425
IMC      906
Unk      186
UNK       76
Name: count, dtype: int64

--- Number.of.Engines ---
Number.of.Engines
1.0    13215
2.0     2467
NaN     2098
4.0       66
3.0       25
0.0        5
Name: count, dtype: int64

--- Purpose.of.flight ---
Purpose.of.flight
Personal              9840
NaN                   3056
Instructional         2409
Aerial Application     723
Business               409
Unknown                301
Positioning            268
Skydiving              154
Aerial Observation     147
Other Work Use         121
Name: count, dtype: int64

--- Broad.phase.of.flight ---
Broad.phase.of.flight
NaN            15438
Landing         

In [100]:
#cleaning
#inconsistent case (e.g., “Piston”, “piston”), spacing, variant wording (“Turbo Prop” vs “TurboProp”).
df['Engine.Type'] = (
    df['Engine.Type']
    .astype(str)
    .str.strip()
    .str.title()                               # e.g., 'turbofan' → 'Turbofan'
    .replace({'Nan': pd.NA, 'Na': pd.NA})      # keep proper NaNs
)


Weather.Condition
Normalize to uppercase, replace blanks with NaN.


In [101]:
df['Weather.Condition'] = (
    df['Weather.Condition']
    .astype(str)
    .str.strip()
    .str.upper()                               # standardize to uppercase like 'VMC'/'IMC'
    .replace({'': pd.NA, 'NAN': pd.NA})
)


Number of engines

In [102]:
df['Number.of.Engines'] = pd.to_numeric(df['Number.of.Engines'], errors='coerce').astype('Int64')


purpose of flight 

In [103]:
df['Purpose.of.flight'] = (
    df['Purpose.of.flight']
    .astype(str)
    .str.strip()
    .str.title()                               # consistent casing
    .replace({'Nan': pd.NA, '': pd.NA})
)


Broad phase of flight.

In [104]:
df['Broad.phase.of.flight'] = (
    df['Broad.phase.of.flight']
    .astype(str)
    .str.strip()
    .str.title()
    .replace({
        'Take Off': 'Takeoff',
        'Pick Up': 'Pickup',
        'Taxiing': 'Taxi',
        'Landing Roll': 'Landing',
        'Nan': pd.NA
    })
)


In [105]:
for col in cols_to_check:
    print(f"\nUnique values after cleaning for {col}:")
    print(df[col].dropna().unique()[:10])



Unique values after cleaning for Engine.Type:
['Reciprocating' 'Turbo Jet' 'Unknown' 'Turbo Prop' 'Turbo Fan'
 'Turbo Shaft' 'Geared Turbofan' 'Unk']

Unique values after cleaning for Weather.Condition:
['VMC' 'UNK' 'IMC']

Unique values after cleaning for Number.of.Engines:
<IntegerArray>
[2, 1, 4, 0, 3]
Length: 5, dtype: Int64

Unique values after cleaning for Purpose.of.flight:
['Personal' 'Aerial Application' 'Skydiving' 'Instructional'
 'Public Aircraft' 'Unknown' 'Ferry' 'Positioning' 'Other Work Use'
 'Business']

Unique values after cleaning for Broad.phase.of.flight:
['Landing' 'Descent' 'Standing' 'Maneuvering' 'Cruise' 'Approach' 'Climb'
 'Takeoff' 'Taxi' 'Go-Around']


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [106]:
# 1. Inspect how many non-null values each column has
non_null_counts = df.notna().sum().sort_values(ascending=False)
non_null_counts  # display result

# 2. Keep columns with more than 20,000 non-null values
df = df.loc[:, df.count() > 20000]

# 3. Confirm new shape and remaining columns
df.shape, df.columns.tolist()


((17876, 0), [])

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [107]:
from pathlib import Path

# define output path
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "aviation_cleaned.csv"

# save the cleaned DataFrame
df.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")


Cleaned dataset saved to: data\processed\aviation_cleaned.csv
